# Feature Engineering - Streaks

In [1]:
import pandas as pd
import sys

sys.path.insert(0, "..")

from src.features import calculate_streaks
from src.utils.constants import Columns
from src.utils.types import Gender
from src.datasets.datasets import compact_regular_season_results_per_gender, compact_tourney_results_per_gender
from src.utils.paths import get_data_directory

In [2]:
m_regular = compact_regular_season_results_per_gender(Gender.MEN)
w_regular = compact_regular_season_results_per_gender(Gender.WOMEN)

In [3]:
m_regular = calculate_streaks(m_regular, reset_between_seasons=True, include_current_game=False)

In [9]:
m_regular.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WWinStreak,WLossStreak,LWinStreak,LLossStreak
156455,2019,9,1388,73,1308,58,A,0,2,0,2,0
11856,1988,35,1286,65,1317,55,A,0,1,0,0,3
133001,2014,96,1293,73,1398,65,A,0,0,1,0,1
143089,2016,82,1454,83,1178,80,H,0,1,0,0,3
113071,2010,123,1287,76,1399,47,N,0,3,0,2,0


In [4]:
w_regular = calculate_streaks(w_regular, reset_between_seasons=True, include_current_game=False)

In [8]:
w_regular.sample(5)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WWinStreak,WLossStreak,LWinStreak,LLossStreak
39923,2007,11,3203,72,3420,58,H,0,0,0,0,0
92116,2017,26,3124,104,3177,72,N,0,3,0,4,0
96638,2018,11,3345,79,3141,69,A,0,0,0,0,0
3429,1998,117,3380,87,3411,58,A,0,1,0,0,1
110888,2020,100,3361,67,3307,58,A,0,0,1,1,0


In [5]:
m_regular.to_csv(get_data_directory() / "MRegularSeasonResultsWithStreaks.csv", index=False)
w_regular.to_csv(get_data_directory() / "WRegularSeasonResultsWithStreaks.csv", index=False)

## Additional Feature Engineering - Ideas

Use streaks to create features for prediction models.

In [ ]:
# Create sample data with scores
games_with_scores = pd.DataFrame(
    {
        Columns.SEASON: [2024] * 10,
        Columns.DAY_NUM: range(1, 11),
        Columns.WTEAM_ID: [101, 101, 102, 101, 102, 103, 101, 102, 103, 101],
        Columns.LTEAM_ID: [102, 103, 101, 104, 103, 101, 102, 104, 102, 103],
        Columns.WSCORE: [75, 82, 68, 91, 77, 85, 72, 88, 80, 76],
        Columns.LSCORE: [70, 78, 65, 85, 72, 80, 70, 84, 75, 71],
    }
)

# Calculate streaks
features = calculate_streaks(games_with_scores, include_current_game=False)

# Create derived features
features["StreakDiff"] = features["WWinStreak"] - features["LWinStreak"]
features["WinnerHotStreak"] = (features["WWinStreak"] >= 3).astype(int)
features["LoserColdStreak"] = (features["LLossStreak"] >= 3).astype(int)
features["CombinedMomentum"] = (
    features["WWinStreak"] + features["WLossStreak"] + features["LWinStreak"] + features["LLossStreak"]
)

print("Features for modeling:")
display(
    features[
        [
            "DayNum",
            "WTeamID",
            "LTeamID",
            "WWinStreak",
            "LWinStreak",
            "StreakDiff",
            "WinnerHotStreak",
            "LoserColdStreak",
            "CombinedMomentum",
        ]
    ]
)